# 20. Does the structural detector survive the paper's own adversary?

Notebook 12 built a black-box severity-maximizing adversary: for each of 91 stratified injections, four paraphrases that keep the actionable instruction but read as ordinary prose; the lowest-scoring variant per detector was selected and executed against Qwen2.5-3B to check it still works. Those rewrites, the released detectors' scores on them, and the target outputs are cached on Drive. This notebook scores the same rewrites with the structural detector and asks three things:

1. **Evasion.** Static versus adaptive FNR for the structural models at the document-calibrated threshold, next to the released detectors at both their document-calibrated threshold and the paper's original deepset threshold (the latter reproduces section 4.11, which doubles as an alignment check).
2. **Transfer.** Of the rewrites the paper manufactured as working confident misses against ProtectAI and Prompt-Guard, what fraction does the structural detector catch?
3. **Own adaptive selection.** The variant the structural detector scores lowest was executed only if it coincides with one of the executed variants; the manufactured rate is reported on that covered subset, with the coverage stated. A full adaptive run against the structural detector needs executing its own selections and is left as the stated limitation.

The subset is reconstructed in notebook 12's order and verified against the cached scores before anything else runs. Judge verdicts are recomputed from the cached outputs (GPU, Qwen2.5-3B in 4-bit, a few minutes) and cached this time.

**Outputs.** `reports/adversary_transfer_evasion.csv`, `reports/adversary_transfer_manufactured.csv`, `figures/adversary_transfer.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes
os.makedirs('reports', exist_ok=True); os.makedirs('figures', exist_ok=True)
import torch; print('Session ready. GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## Stage A. Module

In [ ]:
%%writefile src/adversary_transfer.py
"""Transfer of the notebook-12 severity-maximizing adversary to the structural detector.

Reconstructs the 91-attack stratified subset and the flat variant list (original + four rewrites per attack)
in the exact order notebook 12 used, verifies that order against the cached released-detector scores, and
provides the group-minimum (adaptive) selection and the layout of the cached execution outputs.
"""
from __future__ import annotations
import numpy as np
import pandas as pd


def reconstruct_sub(bip, per_category=13, random_state=0):
    """Same rows in the same order as notebook 12's stratified subsample, with the original bip row index kept."""
    atk = bip[bip.label == 1].copy().reset_index().rename(columns={"index": "bip_index"})
    parts = [d.sample(min(len(d), per_category), random_state=random_state) for _, d in atk.groupby("meta", sort=True)]
    sub = pd.concat(parts).reset_index(drop=True)
    sub["instruction"] = sub.text.map(lambda t: t.split("\n")[-1].strip())
    return sub


def flat_variants(sub, REW):
    """variants list, owner array, and index of each attack's original within the flat list."""
    variants, owner, first = [], [], []
    for i in range(len(sub)):
        first.append(len(variants)); variants.append(sub.text.iloc[i]); owner.append(i)
        for r in REW[i]:
            variants.append(r); owner.append(i)
    return variants, np.array(owner), np.array(first)


def alignment_check(p_flat, first, p_bip_full, bip_index, atol=1e-2, frac_ok=0.9):
    """The originals in the flat list were scored by the same model as the BIPIA panel; their scores must agree."""
    a = np.asarray(p_flat)[first]; b = np.asarray(p_bip_full)[np.asarray(bip_index)]
    d = np.abs(a - b)
    ok = float((d < atol).mean())
    return dict(max_abs_diff=float(d.max()), frac_within_tol=ok, passed=bool(ok >= frac_ok))


def group_min_index(scores, owner):
    scores = np.asarray(scores); n = owner.max() + 1
    return np.array([np.where(owner == i)[0][int(np.argmin(scores[owner == i]))] for i in range(n)])


def exec_layout(n, source_tags=("protectai_v2", "prompt_guard_2")):
    """Index into exec_out.npy: originals first, then the selected rewrite per source detector, in that order."""
    lay = {"orig": np.arange(n)}
    for k, tag in enumerate(source_tags):
        lay[tag] = np.arange(n) + n * (k + 1)
    return lay


## Stage B. Reconstruct the subset and verify alignment against the cached scores

In [ ]:
%%writefile src/adversary_transfer.py
"""Transfer of the notebook-12 severity-maximizing adversary to the structural detector.

Recovers which original attack each cached rewrite group paraphrases, verifies that recovery against the cached
released-detector scores, and provides the flat variant layout, the group-minimum (adaptive) selection and the
layout of the cached execution outputs.
"""
from __future__ import annotations
import numpy as np
import pandas as pd


def recover_subset(bip, REW, P5=None, PB=None, lam=1.0, min_margin=0.02):
    """Recover which original attack each rewrite group paraphrases. Two independent fingerprints: content-word
    overlap between the four rewrites and each candidate original, and (optionally) agreement between the cached
    flat scores of the group's original and the candidate's BIPIA panel score, for each released detector.
    Solved as a one-to-one assignment so no original is claimed twice. Independent of any random generator."""
    import re
    from scipy.optimize import linear_sum_assignment
    atk = bip[bip.label == 1].copy().reset_index().rename(columns={"index": "bip_index"})
    tok = lambda t: set(re.findall(r"[a-z0-9]{3,}", str(t).lower()))
    cand = [tok(t) for t in atk.text]
    S = np.zeros((len(REW), len(atk)))
    for i in range(len(REW)):
        rt = set().union(*[tok(r) for r in REW[i]]) if len(REW[i]) else set()
        S[i] = [len(rt & c) / max(1, len(rt | c)) for c in cand]
    cost = -S.copy()
    if P5 is not None and PB is not None:
        counts = np.array([1 + len(r) for r in REW]); first = np.r_[0, np.cumsum(counts)[:-1]]
        for t in P5:
            a = np.asarray(P5[t])[first][:, None]; b = np.asarray(PB[t])[atk.bip_index.values][None, :]
            cost += lam * np.abs(a - b)
    gi, cj = linear_sum_assignment(cost)
    rows = []
    for i, j in zip(gi, cj):
        others = np.delete(-cost[i], j)
        rows.append(dict(group=int(i), bip_index=int(atk.bip_index.iloc[j]), meta=atk.meta.iloc[j], text=atk.text.iloc[j],
                         sim=float(S[i, j]), margin=float(-cost[i, j] - others.max())))
    sub = pd.DataFrame(rows).sort_values("group").reset_index(drop=True)
    sub["instruction"] = sub.text.map(lambda t: t.split("\n")[-1].strip())
    sub["unique"] = ~sub.bip_index.duplicated(keep=False)
    sub["confident"] = sub.margin >= min_margin
    return sub


def flat_variants(sub, REW):
    """variants list, owner array, and index of each attack's original within the flat list."""
    variants, owner, first = [], [], []
    for i in range(len(sub)):
        first.append(len(variants)); variants.append(sub.text.iloc[i]); owner.append(i)
        for r in REW[i]:
            variants.append(r); owner.append(i)
    return variants, np.array(owner), np.array(first)


def alignment_check(p_flat, first, p_bip_full, bip_index, atol=2e-3, frac_ok=0.95):
    """The originals in the flat list were scored by the same model as the BIPIA panel; their scores must agree.
    Use it with both released detectors: agreement on two independent scorers at a tight tolerance is strong evidence."""
    a = np.asarray(p_flat)[first]; b = np.asarray(p_bip_full)[np.asarray(bip_index)]
    d = np.abs(a - b)
    ok = float((d < atol).mean())
    return dict(max_abs_diff=float(d.max()), frac_within_tol=ok, passed=bool(ok >= frac_ok))


def group_min_index(scores, owner):
    scores = np.asarray(scores); n = owner.max() + 1
    return np.array([np.where(owner == i)[0][int(np.argmin(scores[owner == i]))] for i in range(n)])


def exec_layout(n, source_tags=("protectai_v2", "prompt_guard_2")):
    """Index into exec_out.npy: originals first, then the selected rewrite per source detector, in that order."""
    lay = {"orig": np.arange(n)}
    for k, tag in enumerate(source_tags):
        lay[tag] = np.arange(n) + n * (k + 1)
    return lay

In [ ]:
import importlib, json, numpy as np, pandas as pd
import adversary_transfer, data_loaders, interval_panel; importlib.reload(adversary_transfer); importlib.reload(data_loaders); importlib.reload(interval_panel)
from adversary_transfer import recover_subset, flat_variants, alignment_check, group_min_index, exec_layout
from data_loaders import load_deepset, load_bipia_categorized, BIPIA_MALICIOUS
from detectors import threshold_at_fpr
from interval_panel import three_thresholds, fnr

bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS); deepset = load_deepset(); yd = deepset.label.values
HOST = bip.label.values == 0
TAGS = ['protectai_v2', 'prompt_guard_2']
LABEL = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)'}
REW = np.load('data/phase5/rewrites.npy', allow_pickle=True); assert len(REW) == 91, f'{len(REW)} rewrite groups, expected 91'
P5 = {t: np.load(f'data/phase5/score_{t}.npy') for t in TAGS}
PB = {t: np.load(f'data/score_{t}_bipia_cat.npy') for t in TAGS}
n_flat = int(sum(1 + len(r) for r in REW))
for t in TAGS: assert len(P5[t]) == n_flat, f'{t}: {len(P5[t])} cached scores vs {n_flat} variants'

SUB = recover_subset(bip, REW, P5=P5, PB=PB)
print('recovered subset: unique originals', int(SUB.unique.sum()), '/ 91 | confident', int(SUB.confident.sum()), '/ 91 | content similarity mean %.2f min %.2f | margin min %.3f' % (SUB.sim.mean(), SUB.sim.min(), SUB.margin.min()))
print('category count per 13-group block (notebook 12 built the subset in sorted-category blocks; expect all 1):', [int(SUB.meta.iloc[k*13:(k+1)*13].nunique()) for k in range(7)])
assert SUB.unique.all(), 'two rewrite groups mapped to one original; inspect SUB'
VAR, OWNER, FIRST = flat_variants(SUB, REW); print('variants:', len(VAR))
for t in TAGS:
    chk = alignment_check(P5[t], FIRST, PB[t], SUB.bip_index.values)
    print(f'alignment ({LABEL[t]}, originals in flat list vs BIPIA panel):', chk)
    if not chk['passed']: raise SystemExit('Recovered ownership does not agree with the cached scores. Stop here.')

## Stage C. Score the variants with the structural models (cached)

In [ ]:
import structural_train; importlib.reload(structural_train)
from structural_train import score_texts
CFGS = ['full', 'ablation_no_twins', 'full_sizematched']
SS = {}
for cfg in CFGS:
    fp = f'data/phase5/score_structural_{cfg}.npy'
    if not os.path.exists(fp):
        mdir = f'data/structural/model_{cfg}/best'; assert os.path.exists(os.path.join(mdir, 'config.json')), f'{mdir} missing: run notebook 18'
        print('scoring', cfg); np.save(fp, score_texts(mdir, VAR))
    SS[cfg] = np.load(fp); assert len(SS[cfg]) == len(VAR)
print('scored:', list(SS))

## Stage D. Evasion: static versus adaptive FNR

In [ ]:
def t_doc(scores_bipia):    # document-calibrated guaranteed threshold on the 778 BIPIA hosts
    return three_thresholds(scores_bipia[HOST])['guar']
T = {}
for cfg in CFGS: T[f'structural ({cfg})'] = {'doc': t_doc(np.load(f'data/structural/scores/{cfg}_bipia.npy'))}
for t in TAGS:
    T[LABEL[t]] = {'doc': t_doc(np.load(f'data/score_{t}_bipia_cat.npy')),
                   'paper': threshold_at_fpr(np.load(f'data/score_{t}_deepset.npy')[yd == 0], 0.01)}
SCORES = {**{f'structural ({c})': SS[c] for c in CFGS}, **{LABEL[t]: P5[t] for t in TAGS}}
SEL = {name: group_min_index(s, OWNER) for name, s in SCORES.items()}
rows = []
for name, s in SCORES.items():
    p_static, p_adapt = s[FIRST], s[SEL[name]]
    for tname, tv in T[name].items():
        rows.append(dict(model=name, threshold=tname, t=tv, mean_p_static=float(p_static.mean()), mean_p_adapt=float(p_adapt.mean()),
                         FNR_static=fnr(p_static, tv), FNR_adapt=fnr(p_adapt, tv), dFNR=fnr(p_adapt, tv) - fnr(p_static, tv)))
EVA = pd.DataFrame(rows); EVA.to_csv('reports/adversary_transfer_evasion.csv', index=False)
pd.set_option('display.width', 220)
print('=== static vs adaptive FNR ===')
print(EVA.round(3).to_string(index=False))
print('\nSection 4.11 reproduction check (paper threshold): Prompt-Guard-2 0.593 -> 0.989, ProtectAI-v2 0.648 -> 0.868')

## Stage E. Downstream success of the executed variants (programmatic where possible, judge otherwise; verdicts cached)

In [ ]:
import attack_success; importlib.reload(attack_success)
from attack_success import programmatic_success, judge_prompt, parse_judge, combine, JUDGE_SYSTEM
OUT = list(np.load('data/phase5/exec_out.npy', allow_pickle=True)); LAY = exec_layout(len(SUB), TAGS)
assert len(OUT) == len(SUB) * (1 + len(TAGS)), f'exec_out has {len(OUT)} outputs, expected {len(SUB) * (1 + len(TAGS))}'
meta = [(src, i, SUB.meta.iloc[i], SUB.instruction.iloc[i]) for src in ['orig'] + TAGS for i in range(len(SUB))]
prog = [programmatic_success(m[2], m[3], o) for m, o in zip(meta, OUT)]
need = [j for j, p in enumerate(prog) if p is None]
jfp = 'data/phase5/judge_verdicts.json'
jd = {int(k): v for k, v in json.load(open(jfp)).items()} if os.path.exists(jfp) else {}
todo = [j for j in need if j not in jd]
print(f'programmatic: {len(OUT) - len(need)} | need judge: {len(need)} | cached verdicts: {len(jd)} | to judge now: {len(todo)}')
JUDGE_OK = True
if todo:
    try:
        from targets import load_lm, generate_batch
        tm, tt = load_lm('Qwen/Qwen2.5-3B-Instruct')
        jout = generate_batch(tm, tt, [judge_prompt(meta[j][3], OUT[j]) for j in todo], system=JUDGE_SYSTEM, max_new_tokens=8, batch_size=8)
        for j, o in zip(todo, jout): jd[j] = bool(parse_judge(o))
        json.dump({str(k): v for k, v in jd.items()}, open(jfp, 'w'))
        del tm, tt; import gc; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        JUDGE_OK = False; print(f'JUDGE UNAVAILABLE ({type(e).__name__}: {str(e)[:160]}); manufactured rates will be skipped')
if JUDGE_OK:
    SUCC = np.array([bool(combine(prog[j], jd.get(j, False))) for j in range(len(OUT))])
    print('ASR by source:', {src: round(float(SUCC[idx].mean()), 3) for src, idx in LAY.items()})

## Stage F. Transfer: does the structural detector catch the manufactured misses?

In [ ]:
MAN = None
if JUDGE_OK:
    rows = []
    for src in TAGS:                                   # the paper's manufactured misses, per source detector
        sel_src = SEL[LABEL[src]]                        # index of the variant that source detector selected
        works = SUCC[LAY[src]]
        for tname in ['paper', 'doc']:
            tv = T[LABEL[src]][tname]; missed = P5[src][sel_src] < tv
            manufactured = missed & works
            r = dict(source_detector=LABEL[src], source_threshold=tname, n=len(SUB), evades=float(missed.mean()), still_works=float(works.mean()),
                     manufactured_rate=float(manufactured.mean()), n_manufactured=int(manufactured.sum()))
            for cfg in CFGS:
                caught = SS[cfg][sel_src] >= T[f'structural ({cfg})']['doc']
                r[f'caught_by_{cfg}_of_manufactured'] = float(caught[manufactured].mean()) if manufactured.sum() else np.nan
                r[f'caught_by_{cfg}_of_evasive'] = float(caught[missed].mean()) if missed.sum() else np.nan
            rows.append(r)
    # the structural detector's own selection: success is known only where it coincides with an executed variant
    exec_idx = {j: LAY['orig'][j] for j in range(len(SUB))}
    for cfg in CFGS:
        sel = SEL[f'structural ({cfg})']; tv = T[f'structural ({cfg})']['doc']
        known, works_known, missed_known = [], [], []
        for i in range(len(SUB)):
            j = sel[i]
            cand = [('orig', FIRST[i])] + [(src, SEL[LABEL[src]][i]) for src in TAGS]
            hit = [src for src, idx in cand if idx == j]
            if hit:
                known.append(i); works_known.append(bool(SUCC[LAY[hit[0]][i]])); missed_known.append(bool(SS[cfg][j] < tv))
        works_known, missed_known = np.array(works_known), np.array(missed_known)
        rows.append(dict(source_detector=f'structural ({cfg}) own selection', source_threshold='doc', n=len(SUB), evades=float((SS[cfg][sel] < tv).mean()),
                         coverage_with_known_success=len(known) / len(SUB),
                         manufactured_rate_on_covered=float((missed_known & works_known).mean()) if len(known) else np.nan, n_covered=len(known)))
    MAN = pd.DataFrame(rows); MAN.to_csv('reports/adversary_transfer_manufactured.csv', index=False)
    print('=== transfer of the paper\'s manufactured confident misses to the structural detector ===')
    print(MAN.round(3).to_string(index=False))
else:
    print('skipped (judge unavailable)')

## Stage G. Figure

In [ ]:
import matplotlib.pyplot as plt
d = EVA[(EVA.threshold == 'doc')].reset_index(drop=True); assert len(d) == len(SCORES), f'{len(d)} rows'
fig, ax = plt.subplots(figsize=(9, 4.2)); x = np.arange(len(d)); w = .38
ax.bar(x - w / 2, d.FNR_static, w, label='static attack'); ax.bar(x + w / 2, d.FNR_adapt, w, label='severity-maximizing rewrite (min of 5 variants)')
ax.set_xticks(x); ax.set_xticklabels(d.model, rotation=15, ha='right', fontsize=8); ax.set_ylabel('FNR at document-calibrated 1 percent FPR'); ax.set_ylim(0, 1.05)
ax.set_title('The notebook-12 adversary against each detector'); ax.legend(fontsize=8); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.savefig('figures/adversary_transfer.png', dpi=150); plt.show()

## Stage H. Log and commit

In [ ]:
from reslog import log_result
summary = 'Evasion:\n' + EVA.round(3).to_string(index=False) + ('\n\nManufactured / transfer:\n' + MAN.round(3).to_string(index=False) if MAN is not None else '\n\n(manufactured skipped: judge unavailable)')
log_result('nb20: notebook-12 adversary transferred to the structural detector', summary, csv_df=EVA, csv_name='adversary_transfer_evasion.csv')
print(summary)

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
                    "nb20: severity-maximizing adversary transferred to the structural detector (evasion, manufactured-miss transfer, judge verdicts cached); add src/adversary_transfer.py"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)